In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("https://arxiv.org/pdf/1706.03762")
docs = loader.load()
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(docs)

from google import genai

client = genai.Client(api_key='')

result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=texts[0].page_content
)

print(result.embeddings)

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="chroma_db"
)
'''vector_store.add_documents(texts)

print("Documents added successfully!")'''
retriever=vector_store.as_retriever()
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions based on the provided context.

Previous conversation:
{chat_history}

Context:
{context}

Current question:
{question}

Answer the question using the context and previous conversation.
If the answer cannot be found in the context, say that you don't know.

Answer:
""")
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="poolside/laguna-s-2.1:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER"],
)
# %%
def format_history(messages):

    if not messages:
        return "No previous conversation."

    return "\n".join(
        f"{message.type}: {message.content}"
        for message in messages
    )
from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": RunnableLambda(
            lambda x: format_docs(
                retriever.invoke(
                    x["question"]
                )
            )
        ),

        "question": RunnableLambda(
            lambda x: x["question"]
        ),

        "chat_history": RunnableLambda(
            lambda x: x["chat_history"]
        )
    }

    | prompt
    | llm
    | StrOutputParser()
)
from langchain_community.chat_message_histories import SQLChatMessageHistory

history = SQLChatMessageHistory(
    session_id="user_123",
    connection="sqlite:///chat_history.db"
)
from langchain_core.messages import HumanMessage, AIMessage

history.add_message(
    HumanMessage(question)
)

history.add_message(
    AIMessage(answer)
)

'''history = SQLChatMessageHistory(
    session_id="user_123",
    connection="sqlite:///chat_history.db"
)

for msg in history.messages:
    print(msg.content)'''
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage


DATABASE_URL = "sqlite:///chat_history.db"


def format_history(messages):

    if not messages:
        return "No previous conversation."

    return "\n".join(
        f"{message.type}: {message.content}"
        for message in messages
    )


def ask_rag(question, session_id="user_123"):

    # -----------------------------------
    # 1. Open the conversation history
    # -----------------------------------

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    # -----------------------------------
    # 2. Get previous messages
    # -----------------------------------

    previous_messages = history.messages

    # -----------------------------------
    # 3. Convert history to text
    # -----------------------------------

    chat_history = format_history(
        previous_messages
    )

    # -----------------------------------
    # 4. Run RAG
    # -----------------------------------

    answer = rag_chain.invoke({
        "question": question,
        "chat_history": chat_history
    })

    # -----------------------------------
    # 5. Save user question
    # -----------------------------------

    history.add_message(
        HumanMessage(
            content=question
        )
    )

    # -----------------------------------
    # 6. Save AI answer
    # -----------------------------------

    history.add_message(
        AIMessage(
            content=answer
        )
    )

    # -----------------------------------
    # 7. Return answer
    # -----------------------------------

    return answer
# %%
def show_previous_chats(session_id="user_123"):

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    messages = history.messages

    if not messages:
        print("No previous chats found.")
        return

    print("=" * 60)
    print(f"CHAT HISTORY: {session_id}")
    print("=" * 60)

    for message in messages:

        if message.type == "human":

            print("\n🧑 You:")
            print(message.content)

        elif message.type == "ai":

            print("\n🤖 AI:")
            print(message.content)

        print("-" * 60)

show_previous_chats()
# %%
history = SQLChatMessageHistory(
    session_id="user_123",
    connection=DATABASE_URL
)

print("Number of messages:", len(history.messages))

for message in history.messages:
    print(
        f"{message.type}: {message.content}"
    )
# %%
def clear_chat(session_id="user_123"):

    history = SQLChatMessageHistory(
        session_id=session_id,
        connection=DATABASE_URL
    )

    history.clear()

    print(
        f"Chat history cleared for: {session_id}"
    )
# %%
clear_chat()
